In [4]:
#MLR
# QSS 20 Final Project
# Cooper Hindshaw

# 05_MLR.ipynb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_stata("../dataverse_files/UCT_FINAL_CLEAN.dta")

df = df.drop_duplicates(subset="surveyid").copy()


# Keep observations with both baseline and follow-up assets
rep = df[
    ["surveyid",
     "treat",
     "asset_total_ppp0",
     "asset_total_ppp1",
    "treatXsmall",
    "treatXlarge"]
    ].dropna().copy()

# Rename variables to make the code easier to read
rep = rep.rename(
    columns={"asset_total_ppp0": "assets_baseline",
             "asset_total_ppp1": "assets_followup"})

for var in [
    "asset_livestock_ppp0",
    "asset_durable_ppp0",
    "asset_ag_ppp0",
    "asset_trans_ppp0",
    "durable_investment0"]:
    print(var)

reader = pd.io.stata.StataReader(
    "../dataverse_files/UCT_FINAL_CLEAN.dta")

labels = reader.variable_labels()

for var in [
    "asset_livestock_ppp0",
    "asset_durable_ppp0",
    "asset_ag_ppp0",
    "asset_trans_ppp0",
    "durable_investment0"]:
    print(var, ":", labels[var])

reader = pd.io.stata.StataReader(
    "../dataverse_files/UCT_FINAL_CLEAN.dta"
)

labels = reader.variable_labels()

for var in df.columns:
    if any(word in var.lower() for word in
           ["large", "small", "amount", "transfer", "lump", "monthly", "size", "asset"]):
        print(var, ":", labels.get(var))

df[["treat", "treatXlarge", "treatXsmall"]].value_counts()

rep["treatment_group"] = np.select(
    [(df["treat"] == 0),(df["treatXsmall"] == 1),(df["treatXlarge"] == 1)],
    ["Control","Small Transfer","Large Transfer"],default="Other")

rep["treatment_group"].value_counts()


rep["productive_assets_baseline"] = (
    df["asset_livestock_ppp0"]
    + df["asset_ag_ppp0"]
    + df["asset_trans_ppp0"])

rep["productive_assets_followup"] = (
    df["asset_livestock_ppp1"]
    + df["asset_ag_ppp1"]
    + df["asset_trans_ppp1"])

rep["productive_assets_change"]= (rep["productive_assets_followup"] - rep["productive_assets_baseline"])
rep.head()
rep["productive_assets_change"].describe()

# Balboni uses log assets.
# log1p allows households with zero assets to remain in the sample.
rep["log_passets_baseline"] = np.log1p(rep["productive_assets_baseline"])
rep["log_passets_followup"] = np.log1p(rep["productive_assets_followup"])

rep["log_passets_change"]= (rep["log_passets_followup"] - rep["log_passets_baseline"])





asset_livestock_ppp0
asset_durable_ppp0
asset_ag_ppp0
asset_trans_ppp0
durable_investment0
asset_livestock_ppp0 : Value of livestock (USD)
asset_durable_ppp0 : Value of durable goods (USD)
asset_ag_ppp0 : \hspace{0.2cm}Value of agricultural tools (USD)
asset_trans_ppp0 : \hspace{0.2cm}Value of bike/motorbike (USD)
durable_investment0 : Durable Investments
treatXlump : Treatment x Lump sum transfer
treatXmonthly : Treatment x Monthly transfer
treatXlarge : Treatment x Large transfer
treatXsmall : Treatment x Small transfer
treatXmalerecXlump : Treatment x Male recipient x Lump sum transfer
treatXmalerecXsmall : Treatment x Male recipient x Small transfer
treatXlumpXsmall : Treatment x Lump sum transfer x Small transfer
treatXmalerecXlumpXsmall : Treatment x Male recipient x Lump sum transfer x Small transfer
treatXmalerecXlarge : Treatment x Male recipient x Large transfer
treatXlumpXlarge : Treatment x Lump sum transfer x Large transfer
treatXmalerecXlumpXlarge : Treatment x Male recip

In [5]:
import statsmodels.formula.api as smf

model = smf.ols(
    """
    log_passets_followup
    ~ log_passets_baseline
    + treatXsmall
    + treatXlarge
    + log_passets_baseline:treatXsmall
    + log_passets_baseline:treatXlarge
    """,
    data=rep
).fit(cov_type="HC3")

print(model.summary())

                             OLS Regression Results                             
Dep. Variable:     log_passets_followup   R-squared:                       0.085
Model:                              OLS   Adj. R-squared:                  0.082
Method:                   Least Squares   F-statistic:                     22.06
Date:                  Sun, 30 Aug 2026   Prob (F-statistic):           2.39e-21
Time:                          11:38:06   Log-Likelihood:                -2769.1
No. Observations:                  1440   AIC:                             5550.
Df Residuals:                      1434   BIC:                             5582.
Df Model:                             5                                         
Covariance Type:                    HC3                                         
                                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------